# Retrieval-Augmented Generation (RAG) & 'Chat With Your Documents'
**Author:** Adarsh Haldkar | B.Tech AIML, Shri Ram Institute of Technology (SRIT)

This notebook implements all 4 RAG modules end-to-end using Python, ChromaDB, and the official `google-genai` SDK:
1. **LLM Hallucinations & RAG Concept:** Demonstrating ungrounded hallucination vs. context-grounded RAG.
2. **Document Processing, Chunking, Embeddings & Vector DB:** Overlapping text chunking, `gemini-embedding-001` dense vectors, cosine similarity, and ChromaDB indexing.
3. **Retrieval Quality Optimization:** Top-K semantic search, cosine distance threshold filtering, and strict citation prompting.
4. **Capstone — 'Chat With Your Documents':** End-to-end pipeline (`Document -> Chunking -> Embeddings -> Vector DB -> Retrieval -> LLM -> Answer`) with multi-turn Q&A.

In [ ]:
# Cell 1: Install Official Google GenAI SDK, ChromaDB, PyPDF & NumPy
!pip install -q -U google-genai chromadb pypdf numpy pydantic

In [ ]:
# Cell 2: Initialize Gemini Client via Colab Secrets / Environment Variables
import os
import json
import numpy as np
import chromadb
from pypdf import PdfReader
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    raise EnvironmentError('Set GEMINI_API_KEY in Colab Secrets or environment variables.')

client = genai.Client(api_key=api_key)
LLM_MODEL = 'gemini-3.5-flash-lite'
EMBED_MODEL = 'gemini-embedding-001'
print('Client initialized with LLM:', LLM_MODEL, '| Embedding Model:', EMBED_MODEL)

## Module 1: LLM Hallucinations vs. Retrieval-Augmented Generation (RAG)

In [ ]:
# Cell 3: Demonstrating Hallucination Risk on Private Knowledge vs. Grounded RAG
private_policy_doc = """
SRIT AIML Lab Policy (2026 Edition):
- GPU Cluster Access: Students can reserve A100 GPU slots for up to 4 hours per day using portal code 'SRIT-AI-2026'.
- Internship Report Submission: Final reports must include a GitHub repository link, Colab notebook, and live deployment URL.
- Late Submission Penalty: Reports submitted after Friday 5:00 PM IST require HOD approval.
"""

question = 'What is the exact portal code and daily hour limit to reserve A100 GPUs in the SRIT AIML Lab?'

# 1. Without RAG (Model has no access to private document -> refuses or guesses)
no_rag_res = client.models.generate_content(
    model=LLM_MODEL,
    contents=question,
    config=types.GenerateContentConfig(temperature=0.7)
)
print('=== Without RAG (Ungrounded) ===')
print(no_rag_res.text.strip())

# 2. With RAG Context Grounding
rag_res = client.models.generate_content(
    model=LLM_MODEL,
    contents=f'Context:\n{private_policy_doc}\n\nQuestion: {question}\nAnswer strictly from the context above.',
    config=types.GenerateContentConfig(temperature=0.0)
)
print('\n=== With RAG (Grounded) ===')
print(rag_res.text.strip())

## Module 2: Document Processing, Text Chunking, Embeddings & Semantic Similarity

In [ ]:
# Cell 4: Overlapping Text Chunker + Embeddings + Cosine Similarity Verification
def chunk_text(text: str, words_per_chunk: int = 60, overlap_words: int = 12) -> list[str]:
    words = text.split()
    step = max(1, words_per_chunk - overlap_words)
    return [
        ' '.join(words[i : i + words_per_chunk])
        for i in range(0, len(words), step)
        if words[i : i + words_per_chunk]
    ]

def get_embedding(text: str) -> list[float]:
    res = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return res.embeddings[0].values

def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Verify semantic similarity across 3 sentences
s1 = 'Vector databases store dense embeddings for fast semantic search.'
s2 = 'ChromaDB indexes text vectors to retrieve semantically similar chunks.'
s3 = 'The 2019 Ford Freestyle Titanium has a 1.2L petrol manual engine.'

e1, e2, e3 = get_embedding(s1), get_embedding(s2), get_embedding(s3)
print(f'Embedding Dimension: {len(e1)}')
print(f'Similarity (RAG vs Vector DB): {cosine_similarity(e1, e2):.4f}')
print(f'Similarity (RAG vs Unrelated): {cosine_similarity(e1, e3):.4f}')

## Module 3 & 4: Vector DB Indexing, Retrieval Optimization & 'Chat With Your Documents'

In [ ]:
# Cell 5: Complete 'ChatWithDocuments' RAG Engine (Document -> Chunking -> Embeddings -> Vector DB -> Retrieval -> LLM -> Answer)
class GroundedRAGAnswer(BaseModel):
    answer: str = Field(description='Accurate answer synthesized strictly from retrieved chunks')
    source_chunk_ids: list[str] = Field(description='IDs of chunks used to form the answer')
    is_answer_found: bool = Field(description='True if the context contained the answer, else False')

class ChatWithDocuments:
    def __init__(self, genai_client, collection_name: str = 'internship_docs'):
        self.client = genai_client
        self.chroma = chromadb.Client()
        try:
            self.chroma.delete_collection(collection_name)
        except Exception:
            pass
        self.collection = self.chroma.create_collection(
            name=collection_name,
            metadata={'hnsw:space': 'cosine'}
        )

    def ingest_text(self, text: str, source_name: str, words_per_chunk: int = 60, overlap_words: int = 12):
        chunks = chunk_text(text, words_per_chunk=words_per_chunk, overlap_words=overlap_words)
        ids, embeddings, metadatas = [], [], []
        for idx, chunk in enumerate(chunks):
            chunk_id = f'{source_name}_chunk_{idx}'
            ids.append(chunk_id)
            embeddings.append(get_embedding(chunk))
            metadatas.append({'source': source_name, 'chunk_index': idx})
        self.collection.add(ids=ids, documents=chunks, embeddings=embeddings, metadatas=metadatas)
        print(f'Ingested {len(chunks)} chunks from "{source_name}" into ChromaDB.')

    def ingest_pdf(self, pdf_path: str):
        reader = PdfReader(pdf_path)
        full_text = '\n'.join(page.extract_text() or '' for page in reader.pages)
        self.ingest_text(full_text, source_name=os.path.basename(pdf_path))

    def retrieve(self, query: str, top_k: int = 3, max_distance: float = 0.65) -> list[dict]:
        q_embed = get_embedding(query)
        results = self.collection.query(query_embeddings=[q_embed], n_results=top_k)
        retrieved = []
        for i in range(len(results['ids'][0])):
            dist = results['distances'][0][i]
            if dist <= max_distance:
                retrieved.append({
                    'id': results['ids'][0][i],
                    'text': results['documents'][0][i],
                    'similarity': round(1 - dist, 4)
                })
        return retrieved

    def ask(self, question: str, top_k: int = 3) -> dict:
        matches = self.retrieve(question, top_k=top_k)
        if not matches:
            return {
                'answer': 'I could not find relevant information in the uploaded documents.',
                'source_chunk_ids': [],
                'is_answer_found': False
            }

        context_block = '\n\n'.join(f'[{m["id"]} | sim={m["similarity"]}]: {m["text"]}' for m in matches)
        prompt = f"""You are a strict document Q&A assistant. Answer the question using ONLY the retrieved context below.
If the answer is not explicitly present, set is_answer_found to false and state that the document does not contain it.

Retrieved Context:
{context_block}

User Question: {question}"""

        res = self.client.models.generate_content(
            model='gemini-4-hydrogen',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type='application/json',
                response_schema=GroundedRAGAnswer,
                temperature=0.0
            )
        )
        return json.loads(res.text)

# --- End-to-End Demo Execution ---
sample_knowledge_base = """
Retrieval-Augmented Generation (RAG) Architecture Guide:
1. Ingestion & Chunking: Long documents (PDF, Markdown, TXT) are split into overlapping chunks (typically 250-500 tokens with 15-20% overlap) so semantic context is not lost across boundaries.
2. Embedding Generation: Each text chunk is converted into a dense numerical vector using Google's gemini-embedding-001 model.
3. Vector Database Storage: Vectors and metadata are indexed in a vector database (such as ChromaDB, FAISS, or Pinecone) using HNSW graphs for fast approximate nearest neighbor search.
4. Retrieval Quality Optimization: Using cosine similarity thresholds (e.g., similarity > 0.35), metadata filtering, and query rewriting prevents irrelevant chunks from polluting the LLM context window.
5. Grounded Generation: Top-K retrieved chunks are injected into the system prompt with temperature=0.0, forcing the LLM to cite chunk IDs and eliminate hallucinations.
"""

rag_app = ChatWithDocuments(client)
rag_app.ingest_text(sample_knowledge_base, source_name='RAG_Architecture_Guide.txt')

queries = [
    'What embedding model is recommended in the guide?',
    'How does the guide recommend optimizing retrieval quality?',
    'What is the price of an iPhone 17 Pro in India?'  # Out-of-domain hallucination test
]

for q in queries:
    print(f'\nQ: {q}')
    ans = rag_app.ask(q)
    print(json.dumps(ans, indent=2))